# Final Demo: SeismoFinance Buy/Hold/Sell Prototype

This notebook demonstrates the final project pipeline.

It loads the final processed dataset, prepares a sample input, runs a model prediction when available, and converts the model output into a readable prototype Buy/Hold/Sell signal.

This is an academic demo and not financial advice.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Try to detect project root
PROJECT_ROOT = Path.cwd()

# If running from notebooks folder locally/GitHub
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

try:
    from src.config import MODEL_DATASET_PATH, LSTM_MODEL_PATH, SCALER_PATH
    from src.signals import probabilities_to_signal, format_signal_output
    print("Project modules loaded successfully.")

except ModuleNotFoundError:
    print("Project modules not found. Using Colab fallback paths and functions.")

    MODEL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "model_dataset.csv"
    LSTM_MODEL_PATH = PROJECT_ROOT / "models" / "lstm_model.keras"
    SCALER_PATH = PROJECT_ROOT / "models" / "scaler.pkl"

    CLASS_NAMES = {
        0: "SELL",
        1: "HOLD",
        2: "BUY"
    }

    def class_to_signal(class_id):
        return CLASS_NAMES.get(int(class_id), "UNKNOWN")

    def probabilities_to_signal(probabilities):
        probabilities = np.array(probabilities).flatten()

        if len(probabilities) != 3:
            raise ValueError("Expected exactly 3 probabilities: [SELL, HOLD, BUY].")

        predicted_class = int(np.argmax(probabilities))

        return {
            "sell_probability": float(probabilities[0]),
            "hold_probability": float(probabilities[1]),
            "buy_probability": float(probabilities[2]),
            "predicted_class": predicted_class,
            "final_signal": class_to_signal(predicted_class)
        }

    def format_signal_output(signal_result):
        return (
            f"SELL probability: {signal_result['sell_probability']:.2%}\n"
            f"HOLD probability: {signal_result['hold_probability']:.2%}\n"
            f"BUY probability: {signal_result['buy_probability']:.2%}\n"
            f"Final prototype signal: {signal_result['final_signal']}"
        )

Project modules not found. Using Colab fallback paths and functions.


In [3]:
if MODEL_DATASET_PATH.exists():
    df = pd.read_csv(MODEL_DATASET_PATH)
    print("Dataset loaded successfully.")
    print("Shape:", df.shape)
    display(df.head())
else:
    print("Model dataset not found in this environment.")
    print("Expected path:", MODEL_DATASET_PATH)
    print("This is okay in Colab unless the repo/data folder has been uploaded or cloned.")

Model dataset not found in this environment.
Expected path: /content/data/processed/model_dataset.csv
This is okay in Colab unless the repo/data folder has been uploaded or cloned.


In [4]:
if "df" in globals():
    print("Available columns:")
    for col in df.columns:
        print("-", col)
else:
    print("Dataset is not loaded, so columns cannot be displayed yet.")

Dataset is not loaded, so columns cannot be displayed yet.


In [5]:
if "df" in globals() and len(df) > 0:
    sample = df.tail(1)
    print("Selected latest available sample:")
    display(sample)
else:
    sample = None
    print("No sample available because dataset is missing or empty in this environment.")

No sample available because dataset is missing or empty in this environment.


In [6]:
# Temporary demo probabilities.
# Replace this with real model output once the trained LSTM model is available.
# Probability order: [SELL, HOLD, BUY]

demo_probabilities = np.array([0.42, 0.37, 0.21])

signal_result = probabilities_to_signal(demo_probabilities)

print(format_signal_output(signal_result))

SELL probability: 42.00%
HOLD probability: 37.00%
BUY probability: 21.00%
Final prototype signal: SELL


In [7]:
# Real model prediction section.
# This should be activated once Adi provides:
# - models/lstm_model.keras
# - models/scaler.pkl
# - final feature column list

if LSTM_MODEL_PATH.exists() and SCALER_PATH.exists():
    print("Model and scaler found. Real prediction code can be added here.")
else:
    print("Model or scaler not found in this environment.")
    print("Using demo probabilities for now.")

Model or scaler not found in this environment.
Using demo probabilities for now.


## Interpretation

The final output is a prototype signal:

- SELL means the model expects a negative market reaction.
- HOLD means the model expects a neutral market reaction.
- BUY means the model expects a positive market reaction.

This is an academic decision-support prototype and should not be interpreted as financial advice.